[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/66_lora_merge_unmerge_solution.ipynb)

# 🟡 Solution: LoRA Merge / Unmerge

Reference solution for `lora_merge_unmerge`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

class MergeableLoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, r: int = 4,
                 alpha: float = 1.0, bias: bool = True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        self.weight = nn.Parameter(torch.empty(out_features, in_features), requires_grad=False)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        self.merged = False

    def _update(self):
        return (self.lora_B @ self.lora_A) * self.scaling

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.linear(x, self.weight, self.bias)
        if not self.merged:
            out = out + F.linear(x, self._update(), None)
        return out

    def merge(self):
        if not self.merged:
            with torch.no_grad():
                self.weight.add_(self._update())
            self.merged = True

    def unmerge(self):
        if self.merged:
            with torch.no_grad():
                self.weight.sub_(self._update())
            self.merged = False


In [ ]:
# Verify
layer = MergeableLoRALinear(8, 4, r=2, alpha=4)
x = torch.randn(3, 8)
y1 = layer(x)
layer.merge()
y2 = layer(x)
print(torch.allclose(y1, y2, atol=1e-6), layer.merged)
layer.unmerge()
print(layer.merged)


In [ ]:
# Run judge
from torch_judge import check
check('lora_merge_unmerge')
